# Assignment 2: The Dirty Ledger
**Week 2 — Loading, cleaning and data quality**

This notebook audits `teaching_prices_messy.csv` on a working copy. The original file is not overwritten.

**Data note:** The file labels the observations as illustrative teaching data, not actual PBS observations. All prices are therefore treated as illustrative PKR/kg values only.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Keep the original unchanged and work on a copy.
file_path = Path("teaching_prices_messy.csv")
raw = pd.read_csv(file_path, dtype=str)
df = raw.copy()

print("Rows and columns:", df.shape)
print("\nColumn types before cleaning:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nDistinct values:")
for column in ["date", "item", "city", "unit", "data_label"]:
    print(f"{column}: {df[column].nunique(dropna=False)}")


Rows and columns: (97, 6)

Column types before cleaning:
date          object
item          object
city          object
price         object
unit          object
data_label    object
dtype: object

Missing values:
date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64

Exact duplicate rows: 1

Distinct values:
date: 12
item: 5
city: 2
unit: 1
data_label: 1


## 1. Initial diagnosis and planned decisions

The raw file contains 97 rows and 6 columns. The expected design is 12 weekly dates × 4 items × 2 cities = 96 unique combinations. The extra row is an exact duplicate.

Problems identified:
- `price` was read as text because values include a thousands separator (`1,250`), and it also contains a blank.
- One price is `1,250`, which is inconsistent with the paired city value and surrounding observations.
- One price is `9999`, which is inconsistent with the paired city value and surrounding observations.
- One item label is lowercase `oil b`; this is a formatting inconsistency.
- One row's item label is inconsistent with its price series: on 2026-02-05, `oil b`, City 1, price 108.5 aligns with the Staple A series, while Oil B is around 318.
- One exact duplicate is present.
- One price is missing. It is retained as missing rather than guessed.

The two extreme prices and the mislabelled item are corrected only with an audit note, using the repeated six-unit city difference and adjacent weekly pattern. These are reasoned corrections, not externally verified values.


In [2]:
# Add original CSV row numbers for auditability (header is row 1).
df["source_row"] = np.arange(2, len(df) + 2)

# Normalize harmless whitespace and case differences.
for col in ["item", "city", "unit", "data_label"]:
    df[col] = df[col].str.strip()
df["item"] = df["item"].replace({"oil b": "Oil B"})

# Preserve the original price string before parsing.
df["price_original"] = df["price"]
df["price"] = pd.to_numeric(df["price"].str.replace(",", "", regex=False), errors="coerce")

# Add an audit trail for every manual correction or retained issue.
df["cleaning_note"] = ""

# Row 19 in the CSV: 1,250 is inconsistent with City 1 = 103 and the
# repeated City 2 premium of 6.0. Correct to 109.0, but document inference.
df.loc[df["source_row"] == 19, "price"] = 109.0
df.loc[df["source_row"] == 19, "cleaning_note"] = (
    "Corrected 1,250 to 109.0: paired City 1 value is 103.0 and regular "
    "city gap is 6.0; neighbouring weeks support this."
)

# Row 32 in the CSV: 9,999 is inconsistent with paired City 2 = 163.5.
df.loc[df["source_row"] == 32, "price"] = 157.5
df.loc[df["source_row"] == 32, "cleaning_note"] = (
    "Corrected 9,999 to 157.5: paired City 2 value is 163.5 and regular "
    "city gap is 6.0; neighbouring weeks support this."
)

# Row 42 in the CSV: 108.5 follows Staple A, not Oil B (around 318).
df.loc[df["source_row"] == 42, "item"] = "Staple A"
df.loc[df["source_row"] == 42, "cleaning_note"] = (
    "Relabelled oil b as Staple A: value 108.5 continues Staple A series; "
    "Oil B values are around 318."
)

# Row 7 has no price. Do not invent a value.
df.loc[df["source_row"] == 7, "cleaning_note"] = (
    "Price missing; retained as missing because no source value is available."
)

# Remove only exact duplicate observations, keeping the first occurrence.
duplicate_mask = df.duplicated(
    subset=["date", "item", "city", "price_original", "unit", "data_label"],
    keep="first"
)
duplicates = df.loc[duplicate_mask].copy()
df = df.loc[~duplicate_mask].copy()

# Parse dates and sort for a consistent final table.
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["date", "item", "city"]).reset_index(drop=True)

print("Duplicate rows removed:", len(duplicates))
print("Rows after duplicate removal:", len(df))
print("Missing prices retained:", df["price"].isna().sum())


Duplicate rows removed: 1
Rows after duplicate removal: 96
Missing prices retained: 1


## 2. Before-and-after evidence

The notebook preserves `price_original`, `source_row`, and `cleaning_note` so every edit can be traced. The blank price is not imputed. The duplicate is stored separately for audit purposes.


In [3]:
# Compact before/after audit table for all affected rows.
audit = df.loc[df["cleaning_note"] != "", [
    "source_row", "date", "item", "city", "price_original", "price", "cleaning_note"
]].copy()
audit["date"] = audit["date"].dt.strftime("%Y-%m-%d")
display(audit)

print("\nBefore cleaning:", raw.shape)
print("After cleaning:", df.shape)
print("Before exact duplicates:", raw.duplicated().sum())
print("After exact duplicates:", df.duplicated(
    subset=["date", "item", "city", "price_original", "unit", "data_label"]
).sum())
print("Missing prices after cleaning:", df["price"].isna().sum())


,source_row,date,item,city,price_original,price,cleaning_note
3,7,2026-01-01,Pulse C,City 2,NaN,NaN,Price missing; retained as missing because no ...
21,19,2026-01-15,Staple A,City 2,"1,250",109.0,"Corrected 1,250 to 109.0: paired City 1 value ..."
30,32,2026-01-22,Sweetener D,City 1,9999,157.5,"Corrected 9,999 to 157.5: paired City 2 value ..."
44,42,2026-02-05,Staple A,City 1,108.5,108.5,Relabelled oil b as Staple A: value 108.5 cont...



Before cleaning: (97, 6)
After cleaning: (96, 9)
Before exact duplicates: 1
After exact duplicates: 0
Missing prices after cleaning: 1


## 3. Data quality checks

A complete panel should contain one row per date-item-city combination. There should be 96 combinations. The missing price remains a missing observation, but its row remains in the panel.


In [4]:
# Check expected panel structure.
dates = df["date"].nunique()
items = df["item"].nunique()
cities = df["city"].nunique()
expected_rows = dates * items * cities
unique_keys = df[["date", "item", "city"]].drop_duplicates().shape[0]

print("Dates:", dates, "| Items:", items, "| Cities:", cities)
print("Expected combinations:", expected_rows)
print("Actual rows:", len(df))
print("Unique date-item-city keys:", unique_keys)
print("Duplicate date-item-city keys:", df.duplicated(["date", "item", "city"]).sum())

# Show all missing values and final numeric summary.
print("\nMissing values by column:")
print(df.isna().sum())
print("\nPrice summary:")
display(df["price"].describe())

# Export the cleaned file with the original value and audit trail.
df["date"] = df["date"].dt.strftime("%Y-%m-%d")
output_columns = [
    "date", "item", "city", "price", "price_original", "unit",
    "data_label", "source_row", "cleaning_note"
]
df[output_columns].to_csv("2_cleaned_prices.csv", index=False)
duplicates.to_csv("duplicate_rows_removed.csv", index=False)
print("Saved 2_cleaned_prices.csv and duplicate_rows_removed.csv")


Dates: 12 | Items: 4 | Cities: 2
Expected combinations: 96
Actual rows: 96
Unique date-item-city keys: 96
Duplicate date-item-city keys: 0

Missing values by column:
date              0
item              0
city              0
price             1
unit              0
data_label        0
source_row        0
price_original    1
cleaning_note     0
dtype: int64

Price summary:


count     95.000000
mean     202.126316
std       79.795049
min       99.500000
25%      135.500000
50%      184.500000
75%      255.750000
max      350.000000
Name: price, dtype: float64

Saved 2_cleaned_prices.csv and duplicate_rows_removed.csv


## 4. AI cleaning-code court

| Proposed operation | Verdict | Evidence / reason |
|---|---|---|
| Strip whitespace and standardize `oil b` capitalization | KEEP | Formatting-only inconsistency; does not change the underlying observation. |
| Convert price text to numeric after removing commas | KEEP | `1,250` otherwise prevents numeric analysis; blank becomes `NaN`. |
| Replace 1,250 with 109.0 | MODIFY | Plausible correction supported by paired City 1 = 103.0, the recurring +6 city gap, and adjacent weeks. Must retain original value and note that it is inferred. |
| Replace 9,999 with 157.5 | MODIFY | Plausible correction supported by paired City 2 = 163.5, recurring +6 gap, and neighbouring weeks. Must retain original value and note inference. |
| Relabel row 42 from Oil B to Staple A | MODIFY | Price 108.5 fits Staple A's weekly series; Oil B is around 318. Keep a note because the source does not independently verify the label. |
| Drop exact duplicate | KEEP | Same date, item, city, price, unit and label appears twice; keep one copy and preserve the removed row separately. |
| Fill missing Pulse C / City 2 price | REJECT | No source value is provided. Guessing or imputing would create an observation. Keep it missing and disclose it. |
| Delete every unusual value automatically | REJECT | An outlier is a reason to investigate, not automatic proof of error. |

**Important limitation:** The corrections to 109.0 and 157.5 are strongly supported by internal consistency but cannot be confirmed against an external source. They remain documented assumptions.


In [5]:
# Final compact verification summary.
summary = {
    "raw_rows": len(raw),
    "raw_columns": len(raw.columns),
    "exact_duplicate_rows_removed": len(duplicates),
    "clean_rows": len(df),
    "missing_price_values_retained": int(df["price"].isna().sum()),
    "unique_dates": int(df["date"].nunique()),
    "unique_items": int(df["item"].nunique()),
    "unique_cities": int(df["city"].nunique()),
    "unique_date_item_city_keys": int(df[["date", "item", "city"]].drop_duplicates().shape[0]),
}
summary


{'raw_rows': 97,
 'raw_columns': 6,
 'exact_duplicate_rows_removed': 1,
 'clean_rows': 96,
 'missing_price_values_retained': 1,
 'unique_dates': 12,
 'unique_items': 4,
 'unique_cities': 2,
 'unique_date_item_city_keys': 96}

## 5. Reflection

My decisions differed from a fully automatic cleaning approach in three ways. First, I did not fill the missing price because the file contains no source value for it. Second, I did not simply delete the two extreme prices; I checked their paired-city values and surrounding weeks, then corrected them while preserving the original strings and explaining the inference. Third, I treated the suspicious product label as a possible data-entry error rather than assuming capitalization alone solved it.

The strongest evidence was the repeated six-unit difference between the two cities for the same product and date, alongside the weekly price patterns. Without checking the panel structure, I could have missed the duplicate and the mislabeled product. I could also have made the dataset look complete by inventing the missing value, which would have been less defensible than retaining a visible gap.

The cleaned data is more consistent, but it is not equivalent to verified official data. The file itself states that these are illustrative teaching observations, and the two price corrections plus the product relabeling remain documented inferences.
